In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [2]:
i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.

In [11]:
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\apple"

In [12]:
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    

Found 5763 files belonging to 3 classes.
Using 4611 files for training.


In [13]:
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set

Found 5763 files belonging to 3 classes.
Using 1152 files for validation.


In [14]:
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)

In [15]:
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())

Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']
Train batches: 145
Val batches: 18
Test batches: 18


In [16]:
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)

In [17]:
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [18]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers


In [19]:
num = len(class_names)


In [20]:
base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False

In [21]:
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint

In [22]:
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

In [23]:
x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)

In [24]:
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)

In [25]:
model = models.Model(inputs, outputs)

In [26]:
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [27]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]

In [28]:
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 963ms/step - accuracy: 0.6844 - loss: 0.8591

145/145 ━━━━━━━━━━━━━━━━━━━━ 206s 1s/step - accuracy: 0.8378 - loss: 0.4333 - val_accuracy: 0.8993 - val_loss: 0.2676 - learning_rate: 0.0010
Epoch 2/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9485 - loss: 0.1419

145/145 ━━━━━━━━━━━━━━━━━━━━ 214s 1s/step - accuracy: 0.9521 - loss: 0.1311 - val_accuracy: 0.9722 - val_loss: 0.0882 - learning_rate: 0.0010
Epoch 3/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9642 - loss: 0.1014

145/145 ━━━━━━━━━━━━━━━━━━━━ 211s 1s/step - accuracy: 0.9631 - loss: 0.1000 - val_accuracy: 0.9844 - val_loss: 0.0559 - learning_rate: 0.0010
Epoch 4/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9628 - loss: 0.1022

145/145 ━━━━━━━━━━━━━━━━━━━━ 227s 1s/step - accuracy: 0.9662 - loss: 0.0919 - val_accuracy: 0.9844 - val_loss: 0.0478 - learning_rate: 0.0010
Epoch 5/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 219s 1s/step - accuracy: 0.9740 - loss: 0.0745 - val_accuracy: 0.9844 - val_loss: 0.0486 - learning_rate: 0.0010
Epoch 6/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9712 - loss: 0.0819

145/145 ━━━━━━━━━━━━━━━━━━━━ 212s 1s/step - accuracy: 0.9753 - loss: 0.0695 - val_accuracy: 0.9896 - val_loss: 0.0297 - learning_rate: 0.0010
Epoch 7/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9792 - loss: 0.0641

145/145 ━━━━━━━━━━━━━━━━━━━━ 220s 1s/step - accuracy: 0.9809 - loss: 0.0597 - val_accuracy: 0.9931 - val_loss: 0.0262 - learning_rate: 0.0010
Epoch 8/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9784 - loss: 0.0682

145/145 ━━━━━━━━━━━━━━━━━━━━ 210s 1s/step - accuracy: 0.9813 - loss: 0.0561 - val_accuracy: 0.9965 - val_loss: 0.0170 - learning_rate: 0.0010
Epoch 9/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9836 - loss: 0.0522

145/145 ━━━━━━━━━━━━━━━━━━━━ 211s 1s/step - accuracy: 0.9796 - loss: 0.0588 - val_accuracy: 0.9965 - val_loss: 0.0160 - learning_rate: 0.0010
Epoch 10/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9749 - loss: 0.0665

145/145 ━━━━━━━━━━━━━━━━━━━━ 211s 1s/step - accuracy: 0.9764 - loss: 0.0674 - val_accuracy: 0.9965 - val_loss: 0.0138 - learning_rate: 0.0010


In [29]:
model.save("apple.keras")

In [30]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

18/18 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.9965 - loss: 0.0126
Test Loss: 0.0126
Test Accuracy: 99.65%
